# Phase 7B — Human Review Dashboard & Original Document Access

## Overview

Phase 7B introduced the complete human-review dashboard workflow for the VIGILOX Document Intelligence system.

Before Phase 7B, the backend could:

* Process documents using OCR and LLM extraction.
* Validate evidence and confidence.
* Detect anomalies.
* Generate machine review decisions.
* Store analysis results in PostgreSQL.
* Expose a review queue through the API.
* Persist human review decisions.

However, there were several important operational gaps:

* The original uploaded document was deleted after temporary processing.
* Reviewers could not visually inspect the source document.
* There was no reviewer-facing dashboard.
* Reviewers had no convenient interface to compare extracted values with OCR evidence.
* Human review actions were only available through backend APIs.

Phase 7B solved these problems by adding persistent document storage, image retrieval, a review dashboard, evidence visualization, and reviewer actions.

---

# Phase 7B Objectives

The main objectives were:

1. Permanently store the original uploaded document.
2. Preserve the relationship between the original document and its PostgreSQL record.
3. Allow original documents to be retrieved securely by document ID.
4. Build a browser-based human review dashboard.
5. Display pending review cases.
6. Display the original document beside machine extraction results.
7. Display OCR evidence and confidence for extracted fields.
8. Display anomaly and review-decision information.
9. Allow reviewers to:

   * Approve
   * Reject
   * Correct
10. Preserve machine extraction after human correction.
11. Persist human review decisions and audit events.
12. Remove completed reviews from the pending queue.
13. Validate the entire workflow through an end-to-end test.

---

# Phase 7B Architecture

The final Phase 7B workflow is:

```text
Document Upload
      ↓
Temporary File
      ↓
OCR
      ↓
LLM Structured Extraction
      ↓
Evidence Validation
      ↓
Field Confidence
      ↓
Date / Logical Validation
      ↓
Document Anomaly Validation
      ↓
Machine Review Decision
      ↓
┌────────────────────────────────────┐
│ Persistence Layer                  │
│                                    │
│ PostgreSQL                         │
│ +                                  │
│ Permanent Original Document        │
└────────────────────────────────────┘
      ↓
Review Queue
      ↓
Reviewer Dashboard
      ↓
Original Document + Machine Analysis
      ↓
OCR Evidence + Confidence
      ↓
Approve / Reject / Correct
      ↓
Human Review Record
      ↓
Audit Event
      ↓
Document Removed from Pending Queue
```

---

# Phase 7B.1 — Original Document Storage Service

## Problem

Previously, the upload workflow was:

```text
Upload
  ↓
Temporary File
  ↓
OCR + LLM
  ↓
Database Persistence
  ↓
Temporary File Deleted
```

This meant that after processing, the original source document was no longer available.

A human reviewer therefore could not compare machine output with the actual uploaded document.

---

## Solution

A dedicated storage service was introduced:

```text
src/document_storage_service.py
```

The service provides permanent filesystem storage for uploaded documents.

Default storage structure:

```text
storage/
└── documents/
    └── <document_id>/
        └── original.jpg
```

Equivalent extensions are used for PNG and WEBP documents:

```text
original.jpg
original.png
original.webp
```

---

## Storage Design

The storage location is deterministic.

For example:

```text
document_id:
f56965a0-4757-4a30-a9ab-12178e717113
```

produces:

```text
storage/
└── documents/
    └── f56965a0-4757-4a30-a9ab-12178e717113/
        └── original.jpg
```

No arbitrary user-controlled storage path is used.

---

## Supported Content Types

```text
image/jpeg
image/png
image/webp
```

The storage service maps them to:

```text
image/jpeg → .jpg
image/png  → .png
image/webp → .webp
```

Unsupported content types are rejected.

---

## Storage Operations

`DocumentStorageService` supports:

```text
save_original()
load_original()
original_exists()
delete_document()
get_original_path()
```

---

## Atomic File Storage

Files are first copied to a temporary file within the destination directory.

Then:

```python
os.replace(...)
```

is used to atomically replace the final destination.

This reduces the risk of leaving a partially written original document.

---

## Configurable Storage Root

The default location is:

```text
storage/documents/
```

It can also be configured using:

```text
DOCUMENT_STORAGE_DIR
```

This allows production deployments to move document storage to a dedicated filesystem or mounted volume.

---

## Git Protection

The storage directory should be excluded from source control:

```gitignore
storage/
```

Uploaded identity and licence documents must never be committed to Git.

---

## Phase 7B.1 Test

Test file:

```text
test_phase7b_document_storage.py
```

Verified:

```text
Original document saved                    ✅
Deterministic storage path                 ✅
Original bytes preserved                   ✅
Existence check                            ✅
Document retrieval                         ✅
Unsupported content type rejected          ✅
Missing source file rejected               ✅
Document deletion                          ✅
Repeated deletion safely handled           ✅
```

Final result:

```text
[PASS] PHASE 7B.1 DOCUMENT STORAGE TEST PASSED
```

---

# Phase 7B.2 — Persistence + Storage Integration

## Objective

The next step was connecting filesystem storage with PostgreSQL persistence.

`PersistenceService` was updated so a processed document can now store both:

```text
Database records
+
Original uploaded document
```

---

## Updated Persistence Method

`save_processed_document()` now accepts:

```python
source_path: str | Path | None = None
```

Example:

```python
save_processed_document(
    original_filename="guard.jpg",
    content_type="image/jpeg",
    pipeline_result=result,
    source_path=temp_path,
)
```

---

## Backward Compatibility

`source_path` was intentionally kept optional.

Older internal tests or calls can still use:

```python
save_processed_document(
    original_filename="guard.jpg",
    content_type="image/jpeg",
    pipeline_result=result,
)
```

In that case:

```json
{
  "original_document_stored": false
}
```

When a source file is supplied:

```json
{
  "original_document_stored": true
}
```

---

## Transaction Behavior

The intended workflow became:

```text
Begin PostgreSQL transaction
        ↓
Create document record
        ↓
Generate document_id
        ↓
Store original document
        ↓
Create analysis record
        ↓
Create machine audit event
        ↓
Commit transaction
```

If file storage fails:

```text
File Storage Failure
        ↓
Exception
        ↓
PostgreSQL Transaction Rollback
```

If a later database operation fails after the file was stored, compensating filesystem cleanup removes the stored document.

This prevents database and filesystem state from drifting apart.

---

## Phase 7B.2 Test

Test file:

```text
test_phase7b_persistence_storage_integration.py
```

Verified:

```text
Persistence reports original stored        ✅
Deterministic storage path                  ✅
Original bytes preserved                    ✅
PostgreSQL document record                  ✅
PostgreSQL analysis record                  ✅
Machine audit event                         ✅
Stored original retrievable                 ✅
Legacy persistence still works              ✅
Storage failure causes DB rollback           ✅
Cleanup                                     ✅
```

Final result:

```text
[PASS] PHASE 7B.2 PERSISTENCE + STORAGE INTEGRATION TEST PASSED
```

---

# Phase 7B.3 — Analyze API Storage Integration

## Objective

The production upload API needed to supply its temporary uploaded file to the new persistence layer.

Endpoint:

```http
POST /api/v1/documents/analyze
```

---

## Previous Flow

```text
Upload
  ↓
Temporary File
  ↓
Pipeline
  ↓
Database Persistence
  ↓
Temporary File Deleted
```

---

## Updated Flow

```text
Upload
  ↓
Temporary File
  ↓
Pipeline
  ↓
PersistenceService
  ↓
PostgreSQL
+
Permanent Original Document
  ↓
Temporary File Deleted
```

The key addition was:

```python
source_path=temp_path
```

when calling:

```python
save_processed_document(...)
```

---

## Analyze API Response

The response now also exposes:

```json
{
  "original_document_stored": true
}
```

This confirms that the original source document was successfully copied into permanent storage.

---

## Phase 7B.3 Test

Test file:

```text
test_phase7b_analyze_storage_api.py
```

Verified:

```text
Analyze endpoint HTTP 200                   ✅
API confirms document storage              ✅
Uploaded document permanently stored        ✅
Stored bytes equal upload bytes             ✅
document_id-based path                      ✅
PostgreSQL document verified                ✅
Unsupported file type still HTTP 400        ✅
Cleanup                                     ✅
```

Final result:

```text
[PASS] PHASE 7B.3 ANALYZE API STORAGE TEST PASSED
```

---

# Phase 7B.4 — Original Document Retrieval API

## Objective

Human reviewers needed a secure endpoint to retrieve the original uploaded document.

New endpoint:

```http
GET /api/v1/documents/{document_id}/image
```

---

## Retrieval Logic

The endpoint first queries PostgreSQL.

```text
document_id
    ↓
PostgreSQL document metadata
    ↓
trusted content_type
    ↓
DocumentStorageService
    ↓
deterministic file path
```

The client never supplies a filesystem path.

---

## Response Behavior

For a stored JPEG:

```http
Content-Type: image/jpeg
```

For PNG:

```http
Content-Type: image/png
```

For WEBP:

```http
Content-Type: image/webp
```

The response uses:

```text
Content-Disposition: inline
```

so browsers can render the image directly inside the review interface.

---

## Error Cases

Unknown document:

```text
HTTP 404
Document not found.
```

Existing legacy document without stored image:

```text
HTTP 404
Original document image is not available.
```

Unsupported persisted content type:

```text
HTTP 500
```

because this represents inconsistent stored system state rather than invalid user input.

---

## Phase 7B.4 Test

Test file:

```text
test_phase7b_document_image_api.py
```

Verified:

```text
JPG retrieval HTTP 200                    ✅
JPG bytes preserved                       ✅
JPG Content-Type                          ✅
Inline browser rendering                  ✅
PNG retrieval                             ✅
PNG Content-Type                          ✅
WEBP retrieval                            ✅
WEBP Content-Type                         ✅
Legacy document without image → 404       ✅
Unknown document → 404                    ✅
Deterministic storage lookup              ✅
Cleanup                                   ✅
```

Final result:

```text
[PASS] PHASE 7B.4 ORIGINAL DOCUMENT IMAGE API TEST PASSED
```

---

# Phase 7B.5 — Review Dashboard Foundation

## Objective

A human reviewer interface was added directly to the FastAPI application.

This avoided the need for:

```text
Separate frontend server
Separate deployment
CORS configuration
Additional API host configuration
```

---

## Dashboard Architecture

```text
FastAPI
│
├── /api/v1/...
│
├── /review
│
└── /review/static/...
```

---

## Dashboard Files

```text
src/
└── dashboard/
    ├── index.html
    └── static/
        ├── dashboard.css
        └── dashboard.js
```

Later phases added:

```text
review_detail.html
review_detail.js
```

---

## FastAPI Static Files

Static files are served through:

```python
StaticFiles
```

Mounted at:

```text
/review/static
```

---

## Dashboard Route

```http
GET /review
```

returns:

```text
src/dashboard/index.html
```

---

## Initial Dashboard Components

The first dashboard version included:

* VIGILOX branding
* Sidebar navigation
* Human Review title
* Review Queue page
* Refresh button
* Pending Review summary card
* High Priority card
* Medium Priority card
* Low Priority card
* API health indicator
* Responsive layout

---

## API Health Status

The dashboard calls:

```http
GET /health
```

and displays:

```text
● API Online
```

or:

```text
● API Offline
```

depending on backend availability.

---

## Phase 7B.5 Verification

Browser verification confirmed:

```text
/review page loaded correctly              ✅
Dashboard CSS loaded                       ✅
Dashboard JavaScript loaded                ✅
Sidebar rendered                           ✅
Summary cards rendered                     ✅
Responsive structure                       ✅
API health connection                      ✅
API Online status                          ✅
```

Phase 7B.5 was marked complete after browser verification.

---

# Phase 7B.6 — Review Queue UI

## Objective

The dashboard was connected to the existing backend review queue endpoint:

```http
GET /api/v1/reviews/queue
```

---

## Queue UI

The dashboard now displays:

```text
Pending Reviews
High Priority
Medium Priority
Low Priority
```

using live PostgreSQL data.

---

## Review Table

The queue table includes:

```text
Document
Type
Priority
Reasons
Created
Action
```

Example:

```text
guard_license.jpg
Guard License
MEDIUM
DOCUMENT_EXPIRED
18 Aug 2026
Review
```

---

## Filters

Two filters were added:

```text
Priority
Document Type
```

Priority options:

```text
All Priorities
High
Medium
Low
```

Document type options:

```text
All Document Types
Guard License
SIA Badge
ID Card
```

These call the existing API query parameters:

```http
/api/v1/reviews/queue?priority=MEDIUM
```

and:

```http
/api/v1/reviews/queue?document_type=guard_license
```

or combined:

```http
/api/v1/reviews/queue?priority=MEDIUM&document_type=guard_license
```

---

## Queue States

The interface supports:

```text
Loading
Error
Empty queue
Populated queue
```

---

## XSS Protection

Dynamic strings are passed through HTML escaping before being inserted into generated HTML.

This applies to values such as:

* filenames
* document IDs
* reason codes
* document types

---

## Review Navigation

Initially the Review button opened raw API JSON.

It was later changed to:

```text
/review/{document_id}
```

for the professional reviewer detail page.

---

## Phase 7B.6 Browser Verification

Real PostgreSQL data displayed:

```text
Pending Reviews: 1
High:            0
Medium:          1
Low:             0
```

and:

```text
guard_license.jpg
Guard License
MEDIUM
DOCUMENT_EXPIRED
```

The Review button also resolved the correct document ID.

Phase 7B.6 was therefore completed.

---

# Phase 7B.7 — Document Detail & Evidence UI

## Objective

The reviewer needed to compare the machine analysis with the source evidence rather than inspect raw JSON.

New route:

```http
GET /review/{document_id}
```

Example:

```text
/review/f56965a0-4757-4a30-a9ab-12178e717113
```

---

## New Dashboard Files

```text
src/dashboard/review_detail.html
src/dashboard/static/review_detail.js
```

---

## Review Detail Layout

The page uses a side-by-side design:

```text
┌────────────────────────┬─────────────────────────────┐
│ Original Document      │ Machine Review Decision     │
│                        │                             │
│ Image                  │ Extracted Fields            │
│                        │                             │
│                        │ Confidence                  │
│                        │                             │
│                        │ OCR Evidence                │
│                        │                             │
│                        │ Validation Findings         │
└────────────────────────┴─────────────────────────────┘
```

---

# Original Document Panel

For Phase 7B documents, the page requests:

```http
GET /api/v1/documents/{document_id}/image
```

and displays the returned image.

For legacy records created before Phase 7B:

```text
Original image unavailable
```

is shown instead.

This is considered expected compatibility behavior rather than an application error.

---

# Machine Review Decision

The reviewer can see:

```text
Decision
Priority
Reason Codes
```

Example:

```text
Decision:
REVIEW_REQUIRED

Priority:
MEDIUM

Reason:
DOCUMENT_EXPIRED
```

---

# Extracted Fields

The UI displays fields such as:

```text
Full Name
Licence Number
ID Number
Expiry Date
Date Of Birth
Issue Date
Issuer
```

Each field includes:

```text
Machine Value
Confidence
OCR Evidence
```

Example:

```text
Issuer

TX DPS

Confidence: 98.8%

Evidence:
L15  ISSUED BY TX DPS
```

This allows the reviewer to compare normalized machine extraction with the raw OCR text.

---

# Extraction Structure Compatibility

The dashboard supports both extraction structures used during project development.

Structure A:

```json
{
  "extraction": {
    "full_name": {
      "value": "SAMPLE,JANE"
    }
  }
}
```

Structure B:

```json
{
  "extraction": {
    "fields": {
      "full_name": {
        "value": "SAMPLE,JANE"
      }
    }
  }
}
```

This prevents older database records from becoming unreadable after schema evolution.

---

# OCR Evidence ID Discovery

An important Phase 7B discovery occurred during evidence rendering.

Stored OCR lines looked like:

```json
{
  "bbox": [19, 18, 176, 76],
  "text": "Texas",
  "confidence": 0.9999696016311646
}
```

No explicit:

```text
line_id
```

was persisted.

However, extraction fields referenced evidence such as:

```text
L4
L14
L15
```

Investigation confirmed that the evidence IDs correspond to **zero-based OCR array positions**.

For example:

```text
ocr_lines[4]  → L4
ocr_lines[14] → L14
ocr_lines[15] → L15
```

Verified examples:

```text
L4  → PRINTDATE 01/01/2025
L14 → SAMPLE,JANE
L15 → ISSUED BY TX DPS
```

---

## OCR Lookup Fix

The dashboard therefore reconstructs line IDs using:

```javascript
`L${index}`
```

rather than:

```javascript
`L${index + 1}`
```

Future OCR structures that contain explicit:

```text
line_id
```

or:

```text
id
```

will use those values instead.

---

# Evidence Rendering Verification

The final UI correctly displayed:

```text
Issuer
TX DPS
L15 → ISSUED BY TX DPS
```

```text
Full Name
SAMPLE,JANE
L14 → SAMPLE,JANE
```

```text
Issue Date
2025-01-01
L4 → PRINTDATE 01/01/2025
```

```text
Expiry Date
2026-01-01
L8 → EXPIRES
L9 → 01/01/2026
```

```text
Date Of Birth
1990-01-01
L11 → DOB
L12 → 01/01/1990
```

```text
Licence Number
12345678
L5 → LICENSE
L6 → 12345678
```

This completed the evidence-provenance requirement.

---

# Validation Findings

Document anomalies are shown in a dedicated section.

Example:

```text
DOCUMENT_EXPIRED

Severity:
WARNING

The document has passed its validated expiry date.

Field:
expiry_date
```

---

## Phase 7B.7 Final Status

Verified:

```text
Review detail page                           ✅
Document metadata                            ✅
Machine review decision                      ✅
Priority                                     ✅
Reason codes                                 ✅
Extracted field values                       ✅
Confidence                                   ✅
NULL values                                  ✅
Raw OCR evidence text                        ✅
Zero-based evidence mapping                  ✅
Validation findings                          ✅
Legacy image compatibility                   ✅
Back-to-queue navigation                     ✅
```

Phase 7B.7 was marked complete.

---

# Phase 7B.8 — Approve / Reject / Correct UI

## Objective

The reviewer needed to complete human review directly from the browser.

A Human Review panel was added to the detail page.

---

## Reviewer Inputs

The reviewer supplies:

```text
Reviewer ID
Review Notes
```

Reviewer ID is required.

Notes are optional.

---

## Supported Actions

The UI supports the same actions as `HumanReviewService`:

```text
APPROVE
REJECT
CORRECT
```

---

# HumanReviewService Rules

Allowed actions:

```python
{
    "APPROVE",
    "REJECT",
    "CORRECT",
}
```

Correctable fields:

```python
{
    "document_type",
    "full_name",
    "licence_number",
    "id_number",
    "expiry_date",
    "date_of_birth",
    "issue_date",
    "issuer",
}
```

---

## APPROVE

Payload conceptually:

```json
{
  "reviewer_id": "reviewer-001",
  "action": "APPROVE",
  "notes": "Verified manually."
}
```

Corrections are not allowed.

---

## REJECT

Example:

```json
{
  "reviewer_id": "reviewer-001",
  "action": "REJECT",
  "notes": "Document is invalid."
}
```

Corrections are not allowed.

---

## CORRECT

When the reviewer selects Correct, editable fields are displayed.

Example:

```text
Document Type
Guard License

Full Name
SAMPLE,JANE

Licence Number
12345678

ID Number

Expiry Date
2026-01-01

Date Of Birth
1990-01-01

Issue Date
2025-01-01

Issuer
TX DPS
```

Only changed values are sent.

Example:

```json
{
  "reviewer_id": "reviewer-001",
  "action": "CORRECT",
  "notes": "Corrected expiry and issuer.",
  "corrections": {
    "expiry_date": "2027-01-01",
    "issuer": "TX DPS SECURITY"
  }
}
```

---

# Correction Validation

If the reviewer clicks Submit Corrections without changing anything, the frontend blocks submission:

```text
Change at least one field before submitting corrections.
```

The backend independently enforces the same rule:

```text
CORRECT action requires at least one correction.
```

This provides validation at both UI and service levels.

---

# Review Submission API

All review actions use:

```http
POST /api/v1/documents/{document_id}/reviews
```

Successful submission creates:

```text
human_reviews record
+
HUMAN_REVIEW audit event
```

---

# Machine Extraction Immutability

Human corrections do **not** modify the original machine extraction.

For example:

Machine value:

```text
expiry_date = 2026-01-01
issuer      = TX DPS
```

Human correction:

```text
expiry_date = 2027-01-01
issuer      = TX DPS SECURITY
```

The machine analysis still remains:

```text
expiry_date = 2026-01-01
issuer      = TX DPS
```

The corrections are stored separately in:

```text
human_reviews.corrections
```

This preserves provenance and auditability.

---

# Queue Behavior After Review

A pending review is defined as:

```text
Machine decision = REVIEW_REQUIRED
AND
No human review exists
```

Therefore, after APPROVE, REJECT or CORRECT:

```text
human review exists
        ↓
document automatically disappears
from pending review queue
```

No destructive status overwrite is required.

---

# Phase 7B.8 Operational Test

Test file:

```text
test_phase7b_human_review_actions.py
```

Temporary documents were used so the existing real pending record was not modified.

Verified:

```text
APPROVE test document appears in queue       ✅
APPROVE persisted                            ✅
APPROVE audit persisted                      ✅
APPROVE removed from queue                   ✅

REJECT persisted                             ✅
REJECT audit persisted                       ✅
REJECT removed from queue                    ✅

CORRECT persisted                            ✅
Correction values persisted                  ✅
CORRECT audit persisted                      ✅
CORRECT removed from queue                   ✅

Original machine extraction preserved        ✅

CORRECT without corrections → HTTP 400       ✅
Failed review stays pending                  ✅
```

Final result:

```text
[PASS] PHASE 7B.8 HUMAN REVIEW ACTIONS TEST PASSED
```

---

# Phase 7B.9 — Final Review Dashboard End-to-End Test

## Objective

The final test validated the complete Phase 7B operational workflow rather than individual components.

Test file:

```text
test_phase7b_final_dashboard_e2e.py
```

---

## Full Test Flow

```text
Dashboard HTML
      ↓
Document Upload
      ↓
Temporary File
      ↓
Fake Machine Pipeline
      ↓
PostgreSQL Persistence
      ↓
Original File Storage
      ↓
Review Queue
      ↓
Review Detail Page
      ↓
Document Detail API
      ↓
OCR Evidence
      ↓
Original Image API
      ↓
Human CORRECT
      ↓
PostgreSQL Human Review
      ↓
Audit History
      ↓
Queue Removal
      ↓
Machine Extraction Preservation
```

A fake pipeline was intentionally used because Phase 7B testing was validating review operations rather than external LLM/OCR accuracy.

---

# Final E2E Test Results

The final test produced:

```text
[OK] Final dashboard test services initialized

[PASS] Review dashboard HTML route

[PASS] Document uploaded through analyze API
[PASS] Machine analysis persisted
[PASS] Original document permanently stored
[PASS] PostgreSQL document record verified

[PASS] Document appears in pending review queue
[PASS] Queue exposes machine priority and reason codes

[PASS] Document review detail page available
[PASS] Approve / Reject / Correct controls present

[PASS] Detail API exposes machine extraction
[PASS] OCR evidence available for reviewer provenance

[PASS] Original source image retrievable by reviewer
[PASS] Original image bytes preserved exactly

[PASS] Human CORRECT action accepted by API
[PASS] Human review persisted in PostgreSQL
[PASS] Human corrections persisted

[PASS] Machine review audit present
[PASS] Human review audit present
[PASS] PostgreSQL audit records verified

[PASS] Human-reviewed document removed from pending queue

[PASS] Original machine extraction preserved
```

Final result:

```text
[PASS] PHASE 7B.9 FINAL DASHBOARD END-TO-END TEST PASSED
```

Test data was removed afterward:

```text
[CLEANUP] Phase 7B.9 temporary database and storage data removed.
```

---

# Final Phase 7B API Surface

At the end of Phase 7B, the main reviewer-facing routes are:

## Health

```http
GET /health
```

---

## Analyze Document

```http
POST /api/v1/documents/analyze
```

Performs:

```text
Upload
OCR
LLM extraction
Validation
Machine review decision
PostgreSQL persistence
Original document storage
```

---

## Get Stored Document Analysis

```http
GET /api/v1/documents/{document_id}
```

Returns:

```text
Document metadata
Extraction
OCR lines
Evidence flags
Field confidence
Date validation
Anomaly validation
Machine review decision
```

---

## Get Original Document

```http
GET /api/v1/documents/{document_id}/image
```

Returns the stored source image.

---

## Get Review Queue

```http
GET /api/v1/reviews/queue
```

Optional filters:

```text
priority
document_type
```

---

## Submit Human Review

```http
POST /api/v1/documents/{document_id}/reviews
```

Actions:

```text
APPROVE
REJECT
CORRECT
```

---

## Get Audit History

```http
GET /api/v1/documents/{document_id}/history
```

Returns the document's audit timeline.

---

# Final Dashboard Routes

## Review Queue

```http
GET /review
```

Provides the reviewer queue interface.

---

## Document Review

```http
GET /review/{document_id}
```

Provides:

```text
Original document
Machine decision
Priority
Reason codes
Extracted fields
Confidence
OCR evidence
Validation findings
Human review controls
```

---

# Phase 7B Files Added

```text
src/
├── document_storage_service.py
│
├── dashboard/
│   ├── index.html
│   ├── review_detail.html
│   │
│   └── static/
│       ├── dashboard.css
│       ├── dashboard.js
│       └── review_detail.js
```

---

# Major Files Updated

```text
src/db/persistence_service.py
src/api/main.py
```

The existing Phase 7A queue repository and query logic continued to be used.

---

# Phase 7B Test Files

```text
test_phase7b_document_storage.py

test_phase7b_persistence_storage_integration.py

test_phase7b_analyze_storage_api.py

test_phase7b_document_image_api.py

test_phase7b_human_review_actions.py

test_phase7b_final_dashboard_e2e.py
```

---

# Important Design Decisions

## 1. Original files are stored outside PostgreSQL

PostgreSQL stores metadata and analysis.

The actual binary image is stored on the filesystem.

This avoids unnecessarily storing large binary objects inside the application database.

---

## 2. Storage paths are deterministic

The path is derived from:

```text
document_id
+
content_type
```

rather than storing arbitrary absolute filesystem paths in PostgreSQL.

---

## 3. Human corrections do not overwrite machine output

The machine extraction remains immutable.

Human corrections are stored as separate provenance.

---

## 4. Pending queue is derived from review state

A review is pending when:

```text
REVIEW_REQUIRED
+
no HumanReview record
```

This avoids introducing unnecessary duplicate workflow state.

---

## 5. Legacy documents remain accessible

Records created before original-file storage continue to work.

The dashboard displays:

```text
Original image unavailable
```

instead of failing the entire review page.

---

## 6. Raw OCR evidence is visible to reviewers

Reviewers can compare:

```text
Extracted normalized value
```

against:

```text
Raw OCR evidence
```

Example:

```text
Machine:
TX DPS

Raw OCR:
ISSUED BY TX DPS
```

This is especially useful for assessing normalization and LLM transformation decisions.

---

## 7. Reviewer actions reuse trusted backend decisions

The browser does not submit machine priority or machine reason codes as trusted facts.

The API reloads the stored machine review decision from PostgreSQL before creating the human review record.

This prevents the reviewer client from manipulating machine provenance.

---

# Known Phase 7B Limitations

## 1. OCR line IDs are not explicitly persisted

Current stored OCR records contain:

```json
{
  "text": "...",
  "confidence": 0.99,
  "bbox": [...]
}
```

but no explicit:

```text
line_id
```

The dashboard currently reconstructs evidence IDs using zero-based array positions:

```text
index 0  → L0
index 14 → L14
```

This works with the current pipeline but explicit persistent line IDs would provide stronger long-term provenance.

---

## 2. No reviewer authentication yet

Reviewer identity is currently entered manually:

```text
reviewer-001
```

There is no login/session/authentication layer yet.

---

## 3. Duplicate review submission protection is not yet hardened

The queue excludes documents once a review exists, but production-grade concurrency protection is still required.

For example, two reviewers opening the same item simultaneously could potentially race.

---

## 4. No final corrected-document projection yet

Corrections are stored separately, which is correct for auditability.

However, the dashboard does not yet calculate a convenient final merged view such as:

```text
Machine Extraction
+
Human Corrections
=
Final Reviewed Record
```

---

## 5. Local filesystem storage only

The current implementation uses local disk storage.

Future production deployment may require:

```text
S3
Azure Blob Storage
MinIO
Network storage
```

depending on infrastructure requirements.

---

## 6. No storage retention policy yet

The system does not yet define automatic:

```text
Document retention
Archive
Deletion
Legal hold
Storage cleanup
```

policies.

---

## 7. Dashboard authentication and authorization are absent

Anyone with application access can currently reach:

```text
/review
```

Production deployment must restrict this to authorized reviewers.

---

# Phase 7B Security Characteristics

The Phase 7B design already provides several useful protections:

```text
No arbitrary filesystem path from client            ✅
Document lookup uses trusted PostgreSQL metadata     ✅
Unsupported content types rejected                   ✅
Uploaded source paths never exposed to client        ✅
Original image endpoint uses document_id             ✅
Human corrections validated server-side              ✅
Unsupported correction fields rejected               ✅
Machine review result reloaded from database          ✅
Machine output preserved after human correction       ✅
Audit events created for machine and human actions    ✅
HTML output dynamically escaped where needed          ✅
```

---

# Phase 7B Audit Model

The lifecycle now preserves both machine and human provenance.

Example:

```text
Document
   │
   ├── DocumentAnalysis
   │       │
   │       └── Machine Review Decision
   │
   ├── AuditEvent
   │       └── MACHINE_REVIEW_DECISION
   │
   ├── HumanReview
   │       ├── reviewer_id
   │       ├── action
   │       ├── corrections
   │       └── notes
   │
   └── AuditEvent
           └── HUMAN_REVIEW
```

This means the system can answer:

```text
What did the machine originally extract?

Why did the machine require review?

Who reviewed the document?

What action did the reviewer take?

What values did the reviewer correct?

When did the review occur?

Was the machine output changed?
```

The machine output remains unchanged, while the human review remains independently traceable.

---

# Phase 7B Final Status

```text
Phase 7B.1 — Original Document Storage              ✅ COMPLETE

Phase 7B.2 — Persistence + Storage Integration      ✅ COMPLETE

Phase 7B.3 — Analyze API Storage Integration        ✅ COMPLETE

Phase 7B.4 — Original Document Retrieval API        ✅ COMPLETE

Phase 7B.5 — Review Dashboard Foundation            ✅ COMPLETE

Phase 7B.6 — Review Queue UI                        ✅ COMPLETE

Phase 7B.7 — Document Detail / Evidence UI          ✅ COMPLETE

Phase 7B.8 — Approve / Reject / Correct UI          ✅ COMPLETE

Phase 7B.9 — Final Dashboard End-to-End Test        ✅ COMPLETE
```

# Phase 7B — COMPLETE ✅

At the end of Phase 7B, VIGILOX has moved from a backend-only document intelligence pipeline to an operational **human-in-the-loop document review system** with permanent source-document access, evidence-based review, human corrections, PostgreSQL persistence, and complete audit provenance.
